# GAVE2 V12: R2-V2 + Registered FFA Residual Refinement

This is the clean, resumable three-fold experiment. It keeps the native
`1536 x 1024` canvas, starts every fold from the proven V8 prediction, and
accepts a new task only after fold-stable OOF gains and deterministic
support/skeleton checks. The known-bad sampled path proxy is diagnostic only.

The `8.0+` score is an experimental target, not a promised result. The final
cell emits one candidate only when its worst-fold projection clears a strict
release threshold; otherwise it prints `DO_NOT_SUBMIT`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v12.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
RUN_DIR = DRIVE_BASE / "runs/gave2_v12_safe_3fold"
V8_RUN_DIR = DRIVE_BASE / "runs/gave2_r2v2_v8"
ASSET_DIR = DRIVE_BASE / "assets"
SUBMISSION_ROOT = DRIVE_BASE / "submissions/gave2_v12_safe"

TEAM_ID = "梯度不下降队"
RUN_MINIMA = True
RUN_TASK2 = True
RUN_TASK1 = True
AUTO_DISCONNECT = True
RELEASE_TARGET = 7.95
MINIMA_MIN_SUCCESS_FRACTION = 0.90
EXPECTED_RUNTIME_BUILD_ID = "gave2-v12-r3-kornia083-bf16logits"

assert ARCHIVE_PATH.is_file(), ARCHIVE_PATH
RUN_DIR.mkdir(parents=True, exist_ok=True)
print({"archive": str(ARCHIVE_PATH), "run_dir": str(RUN_DIR), "team_id": TEAM_ID})

## Extract and verify the self-auditing runtime archive

In [ ]:
import hashlib
import json
import os
import shutil
import zipfile

def bytes_sha256(payload):
    return hashlib.sha256(payload).hexdigest()

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    manifest = json.loads(archive.read("archive_manifest.json"))
    if manifest.get("runtime_build_id") != EXPECTED_RUNTIME_BUILD_ID:
        raise RuntimeError(
            "Runtime ZIP does not match this notebook. "
            f"Expected {EXPECTED_RUNTIME_BUILD_ID!r}, found {manifest.get('runtime_build_id')!r}. "
            "Replace MyDrive/MICCAI2026/miccai_v12.zip instead of creating a duplicate file."
        )
    names = set(archive.namelist())
    assert names == set(manifest["members"]) | {"archive_manifest.json"}
    required = {
        "experiments/gave2_v12/train.py",
        "experiments/gave2_v12/predict.py",
        "experiments/gave2_v12/gate.py",
        "experiments/gave2_v12/release.py",
        "experiments/gave2_v12/minima_adapter.py",
        "experiments/gave2_v12/prepare.py",
        "experiments/gave2_v8/predict_r2v2.py",
        "tests/gave2_v12/test_model.py",
        "GAVE2_preliminary/training/images/g_001.png",
        "GAVE2_preliminary/validation/images/g_051.png",
    }
    assert required.issubset(names), sorted(required - names)
    for member, expected in manifest["sha256"].items():
        assert bytes_sha256(archive.read(member)) == expected, member
    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    archive.extractall(WORK_ROOT)

os.chdir(WORK_ROOT)
assert DATA_ROOT.is_dir()
print({
    "verified_members": len(manifest["members"]),
    "archive_kind": manifest["kind"],
    "runtime_build_id": manifest["runtime_build_id"],
})

## Install compatible dependencies and run the release tests

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/gave2_v12/requirements.txt"],
    check=True,
)
import torch
import kornia

assert torch.cuda.is_available(), "Select a GPU runtime"
assert torch.cuda.is_bf16_supported(), "V12 requires a BF16-capable CUDA GPU"
test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/gave2_v12",
        "tests/gave2_v8/test_store_and_fusion.py",
        "tests/gave2_v8/test_submission.py",
        "-q",
    ],
    cwd=WORK_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(test_result.stdout, flush=True)
test_result.check_returncode()
gpu = torch.cuda.get_device_properties(0)
print({
    "torch": torch.__version__,
    "kornia": kornia.__version__,
    "gpu": gpu.name,
    "vram_gib": round(gpu.total_memory / 1024**3, 2),
    "bf16": True,
})

In [ ]:
def run_module(module, *arguments, check=True):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("RUN:", " ".join(command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=WORK_ROOT,
        env=environment,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
        tail = tail[-80:]
    return_code = process.wait()
    result = subprocess.CompletedProcess(command, return_code, stdout="\n".join(tail))
    if check and return_code:
        raise RuntimeError(
            f"{module} failed with return code {return_code}. Last output:\n" + "\n".join(tail)
        )
    return result

R2V2_SOURCE = WORK_ROOT / "external/R2-V2"
R2V2_WEIGHTS = ASSET_DIR / "r2v2"
V8_PREDICTIONS = V8_RUN_DIR / "predictions"
PREPARED_ROOT = RUN_DIR / "prepared_ffa"
MATCHES_ROOT = RUN_DIR / "minima_matches"
FOLD_MANIFEST = RUN_DIR / "fold_manifest.json"

## Acquire the pinned R2-V2 teacher and generate any missing stores

In [ ]:
run_module(
    "experiments.gave2_v8.assets",
    "--source-dir", R2V2_SOURCE,
    "--weights-dir", R2V2_WEIGHTS,
)

for split in ("training", "validation"):
    for model_type in ("av", "bv"):
        run_module(
            "experiments.gave2_v8.predict_r2v2",
            "--data-root", DATA_ROOT,
            "--source-dir", R2V2_SOURCE,
            "--weights-dir", R2V2_WEIGHTS,
            "--output-store", V8_PREDICTIONS / split / model_type,
            "--model-type", model_type,
            "--split", split,
            "--amp", "bf16",
            "--tta",
        )
    run_module(
        "experiments.gave2_v8.fuse",
        "--av-store", V8_PREDICTIONS / split / "av",
        "--bv-store", V8_PREDICTIONS / split / "bv",
        "--output-store", V8_PREDICTIONS / split / "direct",
        "--split", split,
    )

## Register FFA to CFP with pinned MINIMA and conservative QA

In [ ]:
MINIMA_SOURCE = WORK_ROOT / "external/MINIMA"
MINIMA_CHECKPOINT = ASSET_DIR / "minima/minima_loftr.ckpt"

if RUN_MINIMA:
    run_module(
        "experiments.gave2_v12.assets",
        "--source-dir", MINIMA_SOURCE,
        "--checkpoint", MINIMA_CHECKPOINT,
    )
    print("Running one-case MINIMA import and inference preflight", flush=True)
    run_module(
        "experiments.gave2_v12.minima_adapter",
        "--data-root", DATA_ROOT,
        "--source-dir", MINIMA_SOURCE,
        "--checkpoint", MINIMA_CHECKPOINT,
        "--output-root", MATCHES_ROOT,
        "--split", "training",
        "--phase", "FFA_A",
        "--threshold", "0.20",
        "--failure-policy", "error",
        "--limit-cases", "1",
    )
    minima_extraction_failures = []
    for split in ("training", "validation"):
        for phase in ("FFA_A", "FFA_AV"):
            run_module(
                "experiments.gave2_v12.minima_adapter",
                "--data-root", DATA_ROOT,
                "--source-dir", MINIMA_SOURCE,
                "--checkpoint", MINIMA_CHECKPOINT,
                "--output-root", MATCHES_ROOT,
                "--split", split,
                "--phase", phase,
                "--threshold", "0.20",
                "--failure-policy", "identity",
            )
            match_summary = json.loads((MATCHES_ROOT / split / phase / "summary.json").read_text())
            success_fraction = match_summary["successful_cases"] / max(match_summary["cases"], 1)
            print({
                "split": split,
                "phase": phase,
                "successful_match_fraction": round(success_fraction, 3),
                "load_error": match_summary["load_error"],
            })
            if success_fraction < MINIMA_MIN_SUCCESS_FRACTION:
                minima_extraction_failures.append(
                    f"{split}/{phase}: {match_summary['successful_cases']}/{match_summary['cases']} successful"
                )
    if minima_extraction_failures:
        raise RuntimeError(
            "MINIMA extraction gate failed; stop before training. " + "; ".join(minima_extraction_failures)
        )

for split in ("training", "validation"):
    run_module(
        "experiments.gave2_v12.prepare",
        "--data-root", DATA_ROOT,
        "--matches-root", MATCHES_ROOT,
        "--output-root", PREPARED_ROOT,
        "--split", split,
        "--fallback", "identity",
    )
    summary = json.loads((PREPARED_ROOT / split / "summary.json").read_text())
    print({
        "split": split,
        "accepted_registration_fraction": {
            phase: round(value, 3) for phase, value in summary["acceptance_fraction"].items()
        },
        "identity_fallbacks": summary["identity_fallbacks"],
    })

## Create the immutable balanced three-fold split

In [ ]:
run_module(
    "experiments.gave2_v12.folds",
    "--data-root", DATA_ROOT,
    "--output", FOLD_MANIFEST,
    "--seed", "77",
)
folds = json.loads(FOLD_MANIFEST.read_text())
print([len(fold["validation"]) for fold in folds["folds"]])

## Select a full-canvas training profile by measured GPU fit

In [ ]:
PROFILE_CANDIDATES = ((24, 2, False), (20, 2, False), (16, 2, False), (20, 1, False), (16, 1, False))

def select_profile(task):
    for base_channels, batch_size, checkpointing in PROFILE_CANDIDATES:
        arguments = [
            "--data-root", DATA_ROOT,
            "--teacher-store", V8_PREDICTIONS / "training/direct",
            "--task", task,
            "--base-channels", base_channels,
            "--batch-size", batch_size,
            "--steps", 2,
            "--amp", "bf16",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        arguments += ["--activation-checkpointing" if checkpointing else "--no-activation-checkpointing"]
        result = run_module("experiments.gave2_v12.memory_test", *arguments, check=False)
        if result.returncode == 0:
            return {
                "base_channels": base_channels,
                "batch_size": batch_size,
                "activation_checkpointing": checkpointing,
            }
        failure = (result.stdout or "").lower()
        memory_markers = ("out of memory", "outofmemoryerror", "cublas_status_alloc_failed")
        if not any(marker in failure for marker in memory_markers):
            raise RuntimeError(
                f"{task} memory test failed for a non-memory reason; stop after the first profile"
            )
        torch.cuda.empty_cache()
    raise RuntimeError(f"No full-canvas BF16 profile fits {task}")

PROFILES = {"task2": select_profile("task2"), "task1": select_profile("task1")}
(RUN_DIR / "selected_profiles.json").write_text(json.dumps(PROFILES, indent=2))
print(PROFILES)

## Train Task 2 first, then Task 1; resume is automatic

In [ ]:
def train_task(task, epochs, minimum_epochs):
    profile = PROFILES[task]
    for fold in range(3):
        fold_dir = RUN_DIR / "models" / task / f"fold_{fold}"
        arguments = [
            "--data-root", DATA_ROOT,
            "--run-dir", RUN_DIR,
            "--fold-manifest", FOLD_MANIFEST,
            "--teacher-store", V8_PREDICTIONS / "training/direct",
            "--task", task,
            "--fold", fold,
            "--base-channels", profile["base_channels"],
            "--batch-size", profile["batch_size"],
            "--workers", 2,
            "--epochs", epochs,
            "--minimum-epochs", minimum_epochs,
            "--early-stopping-patience", 7,
            "--amp", "bf16",
            "--seed", 77,
            "--activation-checkpointing" if profile["activation_checkpointing"] else "--no-activation-checkpointing",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        if (fold_dir / "config.json").exists():
            arguments.append("--resume")
        run_module("experiments.gave2_v12.train", *arguments)

if RUN_TASK2:
    train_task("task2", epochs=60, minimum_epochs=25)
if RUN_TASK1:
    train_task("task1", epochs=40, minimum_epochs=15)

## Produce three-fold OOF and validation predictions

In [ ]:
RAW_ROOT = RUN_DIR / "predictions/raw"

for task in ("task2", "task1"):
    enabled = RUN_TASK2 if task == "task2" else RUN_TASK1
    if not enabled:
        continue
    for split in ("training", "validation"):
        arguments = [
            "--data-root", DATA_ROOT,
            "--run-dir", RUN_DIR,
            "--fold-manifest", FOLD_MANIFEST,
            "--teacher-store", V8_PREDICTIONS / split / "direct",
            "--output-store", RAW_ROOT / split / task,
            "--task", task,
            "--split", split,
            "--amp", "bf16",
            "--tta",
        ]
        if task == "task2":
            arguments += ["--prepared-root", PREPARED_ROOT]
        run_module("experiments.gave2_v12.predict", *arguments)

## Select only OOF improvements and apply the frozen settings

In [ ]:
SELECTED_ROOT = RUN_DIR / "predictions/selected"
SELECTION_ROOT = RUN_DIR / "selection"
SELECTION_ROOT.mkdir(parents=True, exist_ok=True)

for task in ("task2", "task1"):
    enabled = RUN_TASK2 if task == "task2" else RUN_TASK1
    if not enabled:
        continue
    selection_path = SELECTION_ROOT / f"{task}.json"
    run_module(
        "experiments.gave2_v12.selection", "search",
        "--data-root", DATA_ROOT,
        "--fold-manifest", FOLD_MANIFEST,
        "--teacher-store", V8_PREDICTIONS / "training/direct",
        "--raw-store", RAW_ROOT / "training" / task,
        "--output-config", selection_path,
        "--task", task,
        "--correction-mode", "prune",
        "--corridor-radius", 2,
        "--minimum-pixel-score-gain", 0.20 if task == "task2" else 0.10,
        "--minimum-fold-pixel-score-gain", 0.0,
    )
    for split in ("training", "validation"):
        run_module(
            "experiments.gave2_v12.selection", "apply",
            "--data-root", DATA_ROOT,
            "--teacher-store", V8_PREDICTIONS / split / "direct",
            "--raw-store", RAW_ROOT / split / task,
            "--output-store", SELECTED_ROOT / split / task,
            "--selection", selection_path,
            "--task", task,
            "--split", split,
        )
    selected = json.loads(selection_path.read_text())
    print(task, {"accepted_oof_selection": selected["accepted"], "selected": selected["selected"]})

## Run the independent support and fold-stability gate

In [ ]:
GATE_ROOT = RUN_DIR / "gates"
GATE_ROOT.mkdir(parents=True, exist_ok=True)

for task in ("task2", "task1"):
    enabled = RUN_TASK2 if task == "task2" else RUN_TASK1
    if not enabled:
        continue
    run_module(
        "experiments.gave2_v12.gate",
        "--data-root", DATA_ROOT,
        "--teacher-store", V8_PREDICTIONS / "training/direct",
        "--candidate-store", SELECTED_ROOT / "training" / task,
        "--selection", SELECTION_ROOT / f"{task}.json",
        "--fold-manifest", FOLD_MANIFEST,
        "--task", task,
        "--output", GATE_ROOT / f"{task}.json",
        "--minimum-pixel-score-gain", 0.20 if task == "task2" else 0.10,
        "--minimum-fold-pixel-score-gain", 0.0,
        "--minimum-dice-gain", 0.01,
        "--maximum-sensitivity-drop", 0.025,
        "--maximum-fold-sensitivity-drop", 0.05,
        "--maximum-reassignment-fraction", 0.08,
        "--diagnostic-paths", 0,
    )
    gate = json.loads((GATE_ROOT / f"{task}.json").read_text())
    print(task, {"accepted_release_gate": gate["accepted"], "reasons": gate["reasons"]})

## Freeze Task 3 and build one unambiguous v12_safe candidate

In [ ]:
TASK3_SOURCE = WORK_ROOT / "experiments/gave2_v8/assets/proven_task3"
run_module(
    "experiments.gave2_v12.task3",
    "--data-root", DATA_ROOT,
    "--source", TASK3_SOURCE,
    "--output", RUN_DIR / "task3_frozen_audit.json",
)

run_module(
    "experiments.gave2_v12.submission",
    "--data-root", DATA_ROOT,
    "--output-root", SUBMISSION_ROOT,
    "--team-id", TEAM_ID,
    "--teacher-task1", V8_PREDICTIONS / "validation/direct",
    "--teacher-task2", V8_PREDICTIONS / "validation/direct",
    "--selected-task1", SELECTED_ROOT / "validation/task1",
    "--selected-task2", SELECTED_ROOT / "validation/task2",
    "--task1-gate", GATE_ROOT / "task1.json",
    "--task2-gate", GATE_ROOT / "task2.json",
    "--task3-source", TASK3_SOURCE,
    "--force",
)

submission_manifest = json.loads((SUBMISSION_ROOT / "submission_manifest.json").read_text())
print(json.dumps(submission_manifest, indent=2, ensure_ascii=False))

## Build the source-code package for organizer review

In [ ]:
SOURCE_ZIP = SUBMISSION_ROOT / "GAVE2_V12_source_code.zip"
subprocess.run(
    [
        sys.executable,
        "scripts/build_miccai_v12_archive.py",
        "--output", SOURCE_ZIP,
        "--code-only",
        "--force",
    ],
    cwd=WORK_ROOT,
    check=True,
)
print({"source_code": str(SOURCE_ZIP), "bytes": SOURCE_ZIP.stat().st_size})

## Release decision and optional automatic disconnect

In [ ]:
DECISION_PATH = RUN_DIR / "release_decision.json"
run_module(
    "experiments.gave2_v12.release",
    "--task1-gate", GATE_ROOT / "task1.json",
    "--task2-gate", GATE_ROOT / "task2.json",
    "--submission-root", SUBMISSION_ROOT,
    "--team-id", TEAM_ID,
    "--release-target", RELEASE_TARGET,
    "--output", DECISION_PATH,
)
decision = json.loads(DECISION_PATH.read_text())
print(json.dumps(decision, indent=2, ensure_ascii=False))
if decision["status"] == "DO_NOT_SUBMIT":
    print("DO NOT SUBMIT any V12 ZIP. Keep V8 on the leaderboard.")
else:
    print("Submit exactly one ZIP:", decision["recommended"]["zip"])

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()